In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import os
# import dask.dataframe as dd
import polars as pl

import requests
from bs4 import BeautifulSoup
import pyarrow.parquet as pq
import re
import pyarrow as pa
from IPython.display import display
from io import BytesIO
from pathlib import Path

import duckdb
from glob import glob

import warnings

# Отключаем проверку сертификатов SSL
requests.packages.urllib3.disable_warnings()

In [2]:
work_dir = '/Users/eduard/Documents/Code_projects/ML4RV/ML'

### Загрузка значений индексов

In [3]:
# Определяем пути
notebook_path = os.getcwd()
raw_index_folder = os.path.join(work_dir, 'sergey_t', 'data', 'indicies')
processed_dir = os.path.join(work_dir, 'data', 'indicies', 'processed_data')
output_file = os.path.join(processed_dir, 'stock_indices.csv')

# Создаём папку для сохранения
os.makedirs(processed_dir, exist_ok=True)

In [4]:
# Список для сбора данных
all_dfs = []

print("📊 Собираем данные из файлов...")
for filename in os.listdir(raw_index_folder):
    if filename.endswith('.txt') or filename.endswith('.csv'):
        filepath = os.path.join(raw_index_folder, filename)
        try:
            # Читаем с разделителем ',' и без пропуска строк
            df = pd.read_csv(filepath, sep=',')
        except Exception as e:
            print(f"❌ Ошибка при чтении {filename}: {e}")
            continue

        # Преобразуем <DATE> в datetime
        if '<DATE>' in df.columns:
            df['<DATE>'] = pd.to_datetime(df['<DATE>'], format='%Y%m%d', errors='coerce')
            # Оставляем нужные колонки и переименовываем
            df = df[['<TICKER>', '<DATE>', '<CLOSE>']].copy()
            # Убираем угловые скобки и переводим в нижний регистр
            df.columns = df.columns.str.lower().str.replace('<', '').str.replace('>', '')
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d', errors='coerce')
            # Оставляем нужные колонки и переименовываем
            df = df[['ticker', 'Date', 'Close']].copy()
            # Убираем угловые скобки и переводим в нижний регистр
            df.columns = df.columns.str.lower()
        all_dfs.append(df)

📊 Собираем данные из файлов...


In [5]:
# Объединяем все данные
if not all_dfs:
    raise ValueError("❌ Не удалось прочитать ни один файл. Проверь формат и папку.")

df_combined = pd.concat(all_dfs, ignore_index=True)

# Удаляем дубликаты — оставляем только уникальные пары ticker + date
df_combined.drop_duplicates(subset=['ticker', 'date'], keep='first', inplace=True)

In [6]:
df_combined

,ticker,date,close
0,^CRY,1994-01-03,113.94
1,^CRY,1994-01-04,114.17
2,^CRY,1994-01-05,116.27
3,^CRY,1994-01-06,116.51
4,^CRY,1994-01-07,116.30
...,...,...,...
685027,^CDAX,2025-12-18,2049.61
685028,^CDAX,2025-12-19,2056.96
685029,^CDAX,2025-12-22,2057.44
685030,^CDAX,2025-12-23,2059.56


In [7]:
# Словарь: тикер → страна
index_dict = {
"^AEX": {"name": "AEX", "country": "Netherlands"},
"^AOR": {"name": "ALL ORDINARIES", "country": "Australia"},
"^ATH": {"name": "ATHEX COMP", "country": "Greece"},
"^BEL20": {"name": "BEL20", "country": "Belgium"},
"^BET": {"name": "BET", "country": "Romania"},
"^BUX": {"name": "BUX", "country": "Hungary"},
"^BVP": {"name": "BOVESPA", "country": "Brazil"},
"^CAC": {"name": "CAC40", "country": "France"},
"^DAX": {"name": "DAX", "country": "Germany"},
"^FMIB": {"name": "FTSE MIB", "country": "Italy"},
"^FTM": {"name": "FTSE250", "country": "United Kingdom"},
"^HEX": {"name": "HEX", "country": "Finland"},
"^HSI": {"name": "HANG SENG", "country": "Hong Kong (China)"},
"^IBEX": {"name": "IBEX35", "country": "Spain"},
"^ICEX": {"name": "ICEX", "country": "Iceland"},
"^IPC": {"name": "IPC", "country": "Mexico"},
"^IPSA": {"name": "IPSA", "country": "Chile"},
"^JCI": {"name": "JCI", "country": "Indonesia"},
"^KLCI": {"name": "KLCI", "country": "Malaysia"},
"^KOSPI": {"name": "KOSPI", "country": "South Korea"},
"^MOEX": {"name": "MOEX", "country": "Russia"},
"^MRV": {"name": "MERVAL", "country": "Argentina"},
"^MT30": {"name": "MT30", "country": "Egypt"},
"^NKX": {"name": "NIKKEI225", "country": "Japan"},
"^NZ50": {"name": "NZX50", "country": "New Zealand"},
"^OMXC25": {"name": "OMX COPENHAGEN 25", "country": "Denmark"},
"^OMXR": {"name": "OMX RIGA", "country": "Latvia"},
"^OMXS": {"name": "OMX STOCKHOLM", "country": "Sweden"},
"^OMXT": {"name": "OMX TALLINN", "country": "Estonia"},
"^OMXV": {"name": "OMX VILNIUS", "country": "Lithuania"},
"^OSEAX": {"name": "OSE", "country": "Norway"},
"^PSEI": {"name": "PSEI", "country": "Philippines"},
"^PSI20": {"name": "PSI20", "country": "Portugal"},
"^PX": {"name": "PX", "country": "Czech Republic"},
"^SAX": {"name": "SAX", "country": "Austria"},
"^SET": {"name": "SET", "country": "Thailand"},
"^SHC": {"name": "SSE COMP", "country": "China"},
"^SMI": {"name": "SMI", "country": "Switzerland"},
"^SNX": {"name": "SENSEX", "country": "India"},
"^SOFIX": {"name": "SOFIX", "country": "Bulgaria"},
"^SPX": {"name": "S&P500", "country": "United States"},
"^STI": {"name": "STRAITS TIMES", "country": "Singapore"},
"^TASI": {"name": "TASI", "country": "Saudi Arabia"},
"^TOP40": {"name": "JSE TOP40", "country": "South Africa"},
"^TSX": {"name": "S&P/TSX COMP", "country": "Canada"},
"^TWSE": {"name": "TAIEX", "country": "Taiwan"},
"^XU100": {"name": "XU100", "country": "Turkey"},
'^ISEQ': {'name': 'ISEQ Overall Index', 'country': 'Ireland'},
 '^TA125.TA': {'name': 'TA-125 Index', 'country': 'Israel'},
 '^LUXX': {'name': 'LuxX Index', 'country': 'Luxembourg'},
 '^NGSEALLSHARE': {'name': 'NSE All Share Index', 'country': 'Nigeria'},
 '^CSEGEN': {'name': 'CSE General Index', 'country': 'Cyprus'},
 '^MSE': {'name': 'Macao Stock Exchange Index', 'country': 'Macao'},
 '^ADXGEN': {'name': 'ADX General Index', 'country': 'United Arab Emirates'},
 '^GNRI.QA': {'name': 'QE Index', 'country': 'Qatar'},
 '^BAX': {'name': 'Bahrain All Share Index', 'country': 'Bahrain'},
 '^SBITOP': {'name': 'SBITOP Index', 'country': 'Slovenia'},
 '^KSE100': {'name': 'KSE 100 Index', 'country': 'Pakistan'},
 '^TBSX': {'name': 'Tbilisi Stock Exchange Index', 'country': 'Georgia'},
 '^UX': {'name': 'UX Index', 'country': 'Ukraine'},
 '^NSEASI': {'name': 'NSE All Share Index', 'country': 'Kenya'},
 '^BRVMCOM': {'name': 'BRVM Composite Index', 'country': 'Niger'},
 '^VNINDEX.VN': {'name': 'VN-Index', 'country': 'Vietnam'},
 '^CRBEX': {'name': 'CRBEX Index', 'country': 'Costa Rica'},
 '^ASEGEN': {'name': 'ASE General Index', 'country': 'Jordan'},
 '^CSX': {'name': 'CSX Index', 'country': 'Cambodia'},
 '^MSE20': {'name': 'MSE Top 20 Index', 'country': 'Mongolia'},
 '^CSE': {'name': 'Curacao Stock Exchange Index', 'country': 'Curacao'},
 '^DOMDEX': {'name': 'DOMDEX Index', 'country': 'Dominican Republic'},
 '^BSECOM': {'name': 'BSE Composite Index', 'country': 'Barbados'},
 '^MASI': {'name': 'MASI Index', 'country': 'Morocco'},
 '^LUSEALLSHARE': {'name': 'LUSE All Share Index', 'country': 'Zambia'},
 '^SAX': {'name': 'SAX Index', 'country': 'Slovakia'},
 '^DSEX': {'name': 'DSEX Index', 'country': 'Bangladesh'},
 '^BSE': {'name': 'Baku Stock Exchange Index', 'country': 'Azerbaijan'},
 '^DSEI': {'name': 'DSEI Index', 'country': 'Tanzania'},
 '^BKA.KW': {'name': 'All-Share Index', 'country': 'Kuwait'},
 '^ASPI': {'name': 'All-Share Index', 'country': 'Sri Lanka'},
 '^BSEDOM': {'name': 'BSE Domestic Index', 'country': 'Botswana'},
 '^TTSECOM': {'name': 'TTSE Composite Index',
  'country': 'Trinidad and Tobago'},
 '^JSE': {'name': 'JSE Market Index', 'country': 'Jamaica'},
 '^TUNINDEX': {'name': 'TUNINDEX', 'country': 'Tunisia'},
 '^RSE': {'name': 'Rwanda Stock Exchange Index', 'country': 'Rwanda'},
 '^USEASI': {'name': 'USE All Share Index', 'country': 'Uganda'},
 'CRU.VI': {'name': 'Croatian Traded Index in USD', 'country': 'Croatia'},
 '^IBC': {'name': 'IBC Index', 'country': 'Venezuela'},
 '^GSECOM': {'name': 'GSE Composite Index', 'country': 'Ghana'},
 '^BELEX15': {'name': 'BELEX15 Index', 'country': 'Serbia'},
 '^BSEASI': {'name': 'BSE All Share Index', 'country': 'Lebanon'}
}

In [8]:
index_country_mapping = pd.DataFrame(index_dict).T.reset_index().drop(columns='name')
df_combined = df_combined.merge(
    index_country_mapping,
    left_on='ticker',
    right_on='index',
    how='left'
)

In [9]:
# Проверка на пропущенные страны
missing = df_combined['country'].isnull()
if missing.any():
    unknown_tickers = df_combined[missing]['ticker'].unique()
    print(f"⚠️  Неизвестные тикеры (не сопоставлены со странами): {list(unknown_tickers)}")
    df_combined = df_combined[~missing].copy()
    print(f"🗑️  Удалено строк с неизвестными тикерами: {missing.sum()}")

⚠️  Неизвестные тикеры (не сопоставлены со странами): ['^CRY', '^SHBS', '^SDXP', '^NOMUC', '^RTS', '^NDX', '^DJI', '^TDXP', '^MCFTR', '^RTSTR', '^NDQ', '^DJU', '^DJC', '^DJT', '^UKX', '^MDAX', '^MOEX10', '^CDAX']
🗑️  Удалено строк с неизвестными тикерами: 219503


### Создание фичей

In [10]:
# Год и квартал
df_combined['year'] = df_combined['date'].dt.year
df_combined['quarter'] = df_combined['date'].dt.quarter
df_combined['month'] = df_combined['date'].dt.month


# Конец квартала
df_combined['quarter_end'] = df_combined['date'].dt.to_period('Q').dt.end_time.dt.date

# Конец месяца
df_combined['month_end'] = df_combined['date'].dt.to_period('M').dt.end_time.dt.date

# Конец недели
df_combined['week_end'] = df_combined['date'].dt.to_period('W').dt.end_time.dt.date


# Флаг конца квартала
df_combined['is_quarter_end'] = (df_combined['date'] == df_combined['quarter_end']).astype(int)

# Флаг конца месяца
df_combined['is_month_end'] = (df_combined['date'] == df_combined['month_end']).astype(int)

# Флаг конца недели
df_combined['is_week_end'] = (df_combined['date'] == df_combined['week_end']).astype(int)


# Сортируем
df_combined.sort_values(['ticker', 'country', 'date'], inplace=True)

# Группируем по тикеру, стране, году и кварталу, берём последнюю доступную дату в квартале
df_w = df_combined.groupby(['ticker', 'country', 'year', 'quarter', 'quarter_end', 'week_end']).agg(
    week_end_date=('week_end', 'first'),
    date=('date', 'last'),                      # последняя торговая дата в квартале
    close=('close', 'last')                     # цена в этот день
).reset_index()

# Расчет показателей индекса:
stock_index_wk = df_w.copy(deep=True)

In [11]:
'''
Волатильность
'''
# Недельная волатильность за 2 квартала — какой сейчас уровень шума на рынке?
stock_index_wk['return_wow'] = stock_index_wk.groupby('ticker')['close'].pct_change(1)

stock_index_wk['weekly_vol_2q'] = stock_index_wk.groupby('ticker')['return_wow'].transform(
    lambda x: x.rolling(8, min_periods=6).std()
)

# Годовая (аннуализированная) волатильность — каков общий уровень риска актива в пересчёте на год?
stock_index_wk['vol_1y_annualized'] = stock_index_wk['weekly_vol_2q'] * np.sqrt(52)

# Z-score волатильности - на сколько аномальное поведение волатильности
window = 2*52  # 2 years

stock_index_wk['weekly_vol_zscore'] = stock_index_wk.groupby('ticker')['vol_1y_annualized'].transform(
    lambda x: (x - x.rolling(window, min_periods=52).mean()) / x.rolling(window, min_periods=52).std()
)

In [12]:
'''
Пузырь на рынке
'''
stock_index_wk['ma_20'] = stock_index_wk.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())
stock_index_wk['std_20'] = stock_index_wk.groupby('ticker')['close'].transform(lambda x: x.rolling(20).std())
stock_index_wk['z_score'] = (stock_index_wk['close'] - stock_index_wk['ma_20']) / stock_index_wk['std_20']
stock_index_wk['bubble_signal'] = (stock_index_wk['z_score'].abs() > 2).astype(int)
stock_index_wk = stock_index_wk.drop(columns=['ma_20','std_20','z_score'])

In [13]:
# Группируем по тикеру, стране, году и кварталу, берём последнюю доступную дату в квартале
df_q = stock_index_wk.groupby(['ticker', 'country', 'year', 'quarter']).agg(
    quarter_end_date=('quarter_end', 'first'),
    date=('date', 'last'),                      # последняя торговая дата в квартале
    close=('close', 'last'),                     # цена в этот день
    return_wow=('return_wow','last'),
    weekly_vol_2q=('weekly_vol_2q','last'),
    vol_1y_annualized=('vol_1y_annualized','last'),
    weekly_vol_zscore=('weekly_vol_zscore','last'),
    bubble_signal=('bubble_signal','last'),
).reset_index()

# Расчет показателей индекса:
stock_index_qrt = df_q.copy(deep=True)

In [14]:
'''
Доходности
'''
# Доходность за квартал return_qoq
stock_index_qrt['return_qoq'] = stock_index_qrt.groupby('ticker')['close'].pct_change(1)

# Доходность за год
stock_index_qrt['return_yoy'] = stock_index_qrt.groupby('ticker')['close'].pct_change(4)  # 4 квартала = 1 год

# Средняя годовая доходность за последние 3 года
stock_index_qrt['return_3y_avg'] = stock_index_qrt.groupby('ticker')['return_yoy'].transform(
    lambda x: x.rolling(3, min_periods=2).mean()
)

# Кумулятивный возврат за 3 года
stock_index_qrt['return_3y'] = stock_index_qrt.groupby('ticker')['close'].pct_change(12)  # 12 кварталов

# На сколько % текущая цена выше минимума за 3 года
stock_index_qrt['pct_from_3y_low'] = (
    (stock_index_qrt['close'] - stock_index_qrt.groupby('ticker')['close'].transform(
        lambda x: x.rolling(12, min_periods=8).min()
    )) / stock_index_qrt.groupby('ticker')['close'].transform(
        lambda x: x.rolling(12, min_periods=8).min()
    )
)

# Или: на сколько % от максимума
stock_index_qrt['pct_to_3y_high'] = stock_index_qrt['close'] / stock_index_qrt.groupby('ticker')['close'].transform(
    lambda x: x.rolling(12, min_periods=8).max()
)

'''
Режим рынка
'''
# Возьмём 8-квартальную (2 года)
stock_index_qrt['ma_8q'] = stock_index_qrt.groupby('ticker')['close'].transform(
    lambda x: x.rolling(8, min_periods=5).mean()
)

# Флаг: бычий рынок = цена выше скользящей
stock_index_qrt['market_regime'] = (stock_index_qrt['close'] > stock_index_qrt['ma_8q']).astype(int)

'''
Относительная сила - насколько быстро растёт индекс по сравнению с долгосрочным трендом
'''
# RS = цена / скользящая средняя (например, 8-квартальная)
stock_index_qrt['rs_index'] = stock_index_qrt['close'] / stock_index_qrt['ma_8q']

# Или z-score отклонения от MA
stock_index_qrt['rs_zscore'] = stock_index_qrt.groupby('ticker')['rs_index'].transform(
    lambda x: (x - x.rolling(8, min_periods=5).mean()) / x.rolling(8, min_periods=5).std()
)

'''
Скорость роста (Momentum / Acceleration) -
'''
# Ускорение = изменение квартальной доходности
stock_index_qrt['momentum_accel'] = stock_index_qrt.groupby('ticker')['return_qoq'].transform(
    lambda x: x.diff(1)  # изменение доходности
)

'''
Флаги экстремальных режимов - Бинарные фичи для кризисов, пузырей, стресса.
'''
# Высокая волатильность
stock_index_qrt['high_vol_regime'] = (stock_index_qrt['weekly_vol_zscore'] > 1.5).astype(int)

# Резкое падение (кризис)
stock_index_qrt['sharp_drop'] = (stock_index_qrt['return_qoq'] < -0.20).astype(int)  # падение >20% за квартал

# Новый максимум (пузырь?)
stock_index_qrt['new_high'] = stock_index_qrt.groupby('ticker')['close'].transform(
    lambda x: (x == x.rolling(12, min_periods=8).max()).astype(int)
)

In [15]:
'''
Сезонность рынка (по кварталам)
'''
# Средняя доходность по кварталам за последние 5 лет
seasonality = stock_index_qrt.groupby(['ticker', 'quarter'])['return_qoq'].rolling(5).mean().reset_index()
seasonality = seasonality.groupby(['ticker', 'quarter'])['return_qoq'].mean().rename('seasonal_factor')

stock_index_qrt = stock_index_qrt.merge(seasonality, on=['ticker', 'quarter'], how='left')

In [16]:
# rename mapping countries into Damodaran format (only which differ)
country_names_mapping = {
    'South Korea': 'Korea',
    'United States': 'US'
}

stock_index_qrt['country'] = stock_index_qrt['country'].replace(country_names_mapping)

In [17]:
# Сохраняем
stock_index_qrt.to_csv(output_file, index=False)
print(f"✅ Данные обработаны и сохранены: {output_file}")
print(f"📊 Размер финального фрейма: {stock_index_qrt.shape[0]:,} строк, {stock_index_qrt['ticker'].nunique()} индексов.")

✅ Данные обработаны и сохранены: /Users/eduard/Documents/Code_projects/ML4RV/ML/data/indicies/processed_data/stock_indices.csv
📊 Размер финального фрейма: 8,456 строк, 47 индексов.
